# 🌍 Global Temperature Heatmap & GP Visualization

This notebook provides a template for visualizing the Gaussian Process "beliefs" about global temperatures based on the samples collected in `src/data/results.csv`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Set project root
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"Current working directory: {os.getcwd()}")

: 

## 1. Load Collected Data

In [ ]:
df = pd.read_csv('src/data/results.csv')
print(f"Loaded {len(df)} samples.")
df.head()

## 2. Fit Gaussian Process (Placeholder Model)

We use the sampled points to train a GP that can predict temperatures for any coordinate on Earth.

In [ ]:
if len(df) > 0:
    # X = (lat, lng), y = temp
    X = df[['lat', 'lng']].values
    y = df['temp'].values

    # Define Kernel (RBF is a good starting point for spatial data)
    # Note: For real world use, length_scale should be tuned (Role 3 task)
    kernel = C(1.0) * RBF(length_scale=10.0)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
    
    gp.fit(X, y)
    print("GP Model fitted successfully.")
else:
    print("No data yet! Please run some samples first.")

## 3. Generate Global Predictions

We create a grid of the entire world and ask the GP for its predicted temperature and uncertainty.

In [ ]:
# Create a grid (every 5 degrees)
lat_grid = np.linspace(-90, 90, 36)
lng_grid = np.linspace(-180, 180, 72)
lat_mesh, lng_mesh = np.meshgrid(lat_grid, lng_grid)

grid_points = np.vstack([lat_mesh.ravel(), lng_mesh.ravel()]).T

if len(df) > 0:
    y_pred, sigma = gp.predict(grid_points, return_std=True)
    
    # Reshape for visualization
    grid_df = pd.DataFrame({
        'lat': grid_points[:, 0],
        'lng': grid_points[:, 1],
        'pred_temp': y_pred,
        'uncertainty': sigma
    })
    print("Global predictions generated.")

## 4. Visualize Predicted Temperatures

This map shows where the model *believes* it is hot.

In [ ]:
if 'grid_df' in locals():
    fig = px.density_mapbox(grid_df, lat='lat', lon='lng', z='pred_temp', 
                            radius=20, center=dict(lat=0, lon=0), zoom=0,
                            mapbox_style="carto-positron",
                            title="GP Predicted Temperature Heatmap")
    
    # Add the actual sample points as markers
    fig.add_trace(go.Scattermapbox(
        lat=df['lat'],
        lon=df['lng'],
        mode='markers',
        marker=go.scattermapbox.Marker(size=8, color='black'),
        name='Sampled Points'
    ))
    
    fig.show()
else:
    print("Run previous cells first.")

## 5. Visualize Uncertainty

This map shows where the model is *least confident* (it has high sigma). The BO algorithm should target these areas.

In [ ]:
if 'grid_df' in locals():
    fig = px.density_mapbox(grid_df, lat='lat', lon='lng', z='uncertainty', 
                            radius=20, center=dict(lat=0, lon=0), zoom=0,
                            mapbox_style="carto-positron",
                            title="GP Model Uncertainty (Variance)")
    fig.show()